In [4]:
import ee
import geemap
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. INITIALIZATION
ee.Initialize(project='belgaum-492521')

# 2. SETTINGS
COST_PER_KWH = 7.00
CO2_PER_KWH = 0.8
K_LUMINOUS = 0.0012
belgaum_center = ee.Geometry.Point([74.50, 15.85])
roi = belgaum_center.buffer(12500).bounds()

# --- PART A: THE DATA ENGINE & VISUAL COMPOSITOR ---
print("Step 1: Compositing Layers (Basemap + Boundary + NTL)...")

# Load a dark basemap style (using Google Maps stylized base)
basemap = ee.Image("Mapbox/dark-v10").clip(roi) # Or a dimmed satellite layer

# Create the Boundary Line (White, thickness 2)
boundary_outline = ee.Image().paint(roi, 0, 2).visualize(palette=['#ffffff'], opacity=0.4)

collection = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG") \
    .filterBounds(roi) \
    .filterDate('2024-01-01', '2024-12-31')

def create_final_frame(img):
    # 1. Process NTL Data
    mask = img.select('cf_cvg').gt(0)
    ntl_rad = img.select('avg_rad').updateMask(mask).clip(roi)

    # 2. Convert NTL to RGB (Heatmap style)
    ntl_rgb = ntl_rad.visualize(
        min=0, max=40,
        palette=['black', '#000044', '#000088', '#FFFF00', '#FFFFFF']
    )

    # 3. LAYER STACKING: Start with NTL -> Add Boundary -> Add Title/Text later
    # Note: We blend the boundary outline so it sits on top of the lights
    composite = ntl_rgb.blend(boundary_outline)

    # Calculation Bands (stored for the chart)
    kwh = ntl_rad.multiply(ee.Image.pixelArea()).multiply(K_LUMINOUS).rename('kWh')

    return composite.addBands([kwh, ntl_rad]).copyProperties(img, ['system:time_start'])

processed_coll = collection.map(create_final_frame)

# --- PART B: FISCAL DATA EXTRACTION ---
print("Step 2: Calculating Fiscal Metrics in Millions...")
stats = processed_coll.map(lambda img: ee.Feature(None, {
    'date': img.date().format('MMM YYYY'),
    'kWh': img.reduceRegion(ee.Reducer.sum(), roi, 463).getNumber('kWh')
})).getInfo()['features']

df = pd.DataFrame([f['properties'] for f in stats]).sort_values('date')
df['INR_M'] = (df['kWh'] * COST_PER_KWH) / 1_000_000
df['CO2'] = df['kWh'] * CO2_PER_KWH

# --- PART C: INTERACTIVE BI DASHBOARD (CLEAN WHITE THEME) ---
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df['date'], y=df['kWh'], name="Energy (kWh)",
                         line=dict(color='#2980b9', width=4, shape='spline'), fill='tozeroy',
                         customdata=df['INR_M'],
                         hovertemplate="<b>%{x}</b><br>Energy: %{y:,.0f} kWh<br>Cost: ₹%{customdata:.2f} M<extra></extra>"), secondary_y=False)
fig.add_trace(go.Scatter(x=df['date'], y=df['CO2'], name="CO2 (kg)",
                         line=dict(color='#e74c3c', width=2, dash='dot', shape='spline')), secondary_y=True)
fig.update_layout(title='<b>Belgaum Urban Energy Index</b>', plot_bgcolor='white', paper_bgcolor='white', hovermode="x unified")
fig.show()

# --- PART D: THE "CONTEXTUAL" GIF ---
print("Step 3: Exporting GIF with Burned-In Labels...")
out_gif = "belgaum_2024_final.gif"

# We select the visualized RGB bands for the GIF
geemap.download_ee_video(processed_coll.select(['vis-red', 'vis-green', 'vis-blue']),
                         {'dimensions': 800, 'region': roi, 'framesPerSecond': 1},
                         out_gif)

# Add The "Who, Where, When" Labels
geemap.add_text_to_gif(out_gif, out_gif, xy=('5%', '5%'),
                       text_sequence='BELGAUM MUNICIPAL BOUNDARY (2024)',
                       font_color='cyan', font_size=20)

geemap.add_text_to_gif(out_gif, out_gif, xy=('5%', '90%'),
                       text_sequence=df['date'].tolist(),
                       font_color='white', font_size=32, add_progress_bar=True)

print("\nFinished! You now have:")
print("1. belgaum_2024_final.gif -> Contextual Animation")
print("2. Interactive Plotly Dashboard -> (See above)")

Step 1: Compositing Layers (Basemap + Boundary + NTL)...
Step 2: Calculating Fiscal Metrics in Millions...


Step 3: Exporting GIF with Burned-In Labels...
Generating URL...
Please wait ...
The GIF image has been saved to: /content/belgaum_2024_final.gif

Finished! You now have:
1. belgaum_2024_final.gif -> Contextual Animation
2. Interactive Plotly Dashboard -> (See above)
